In [1]:
!pip install "osl-dynamics[tf]"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.8/8.8 MB 82.0 MB/s eta 0:00:00


In [2]:
!pip install pqdm

In [3]:
#  DATA LOADER & FEATURE ENGINEERING FOR CLASSICAL ML
import os
import pickle
import numpy as np
import pandas as pd
import mne
from mne.preprocessing import Xdawn
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler

INPUT_EPOCHS = "/kaggle/input/data-processing/processed/processed_data.pkl"
OUTPUT_CSV = "/kaggle/working/optimal_xdawn_dataset.csv"

# Cửa sổ P300 chuẩn
TMIN, TMAX = 0.2, 0.6  # Lấy rộng hơn chút để xDAWN học được bối cảnh

def generate_optimal_data():
    print(f"💎 KHỞI TẠO QUY TRÌNH TỐI ƯU HÓA DỮ LIỆU (xDAWN)")
    
    if not os.path.exists(INPUT_EPOCHS):
        print("❌ Thiếu input."); return

    with open(INPUT_EPOCHS, 'rb') as f:
        all_subjects = pickle.load(f)
        
    final_data = []

    for subj_id, epochs in all_subjects.items():
        # Kiểm tra số lượng mẫu
        n_targets = np.sum(epochs.events[:, 2] == 6)
        if n_targets < 5:
            print(f"⚠️ Bỏ qua {subj_id} (Dữ liệu quá ít).")
            continue
            
        print(f"⚡ Đang tối ưu hóa: {subj_id}...")
        
        # TIỀN XỬ LÝ SƠ BỘ
        # Chỉ lấy EEG, loại bỏ kênh rác
        epochs.pick_types(eeg=True)
        
        # Crop dữ liệu để giảm nhiễu biên
        epochs.crop(tmin=TMIN, tmax=TMAX)
        
        # ÁP DỤNG xDAWN (SPATIAL FILTERING)
        # Mục tiêu: Tìm ra 3 "Siêu kênh" đại diện tốt nhất cho P300
        # xDAWN cần biết event nào là Target (6) để tối ưu cho nó
        # Lưu ý: xDAWN là thuật toán có giám sát (supervised), ta cần fit vào epochs
        try:
            xdawn = Xdawn(n_components=3, correct_overlap=False)
            # Fit vào epochs (MNE tự động dùng events để học sự khác biệt)
            xdawn.fit(epochs)
            
            # Biến đổi dữ liệu: 127 kênh -> 3 thành phần xDAWN sạch
            epochs_denoised = xdawn.apply(epochs)['Stimulus/S  6'] # Lấy thành phần liên quan Target
        except Exception as e:
            print(f"   ❌ Lỗi xDAWN: {e}. Dùng dữ liệu gốc.")
            epochs_denoised = epochs
            
        # TRÍCH XUẤT ĐẶC TRƯNG 
        # Bây giờ ta có (n_epochs, 3, n_times)
        X = epochs_denoised.get_data()
        y = epochs_denoised.events[:, 2]
        times = epochs_denoised.times
        
        for i in range(len(X)):
            # Lọc Label: Chỉ lấy Target (1) và Standard (0)
            if y[i] == 6: label = 1
            elif y[i] == 5: label = 0
            else: continue
                
            row = {'subject_id': subj_id, 'epoch_idx': i, 'label': label}
            
            # Extract trên 3 components (Comp_0 là quan trọng nhất)
            for comp_idx in range(X.shape[1]): # Loop qua 3 components
                signal = X[i, comp_idx, :]
                
                # --- CÁC ĐẶC TRƯNG ---
                # Mean Amplitude: Độ lớn trung bình
                row[f'Comp{comp_idx}_Mean'] = np.mean(signal)
                
                # Maximum Amplitude: Đỉnh cao nhất (Amplitude)
                row[f'Comp{comp_idx}_Max'] = np.max(signal)
                
                # Thời điểm đạt đỉnh (tính bằng ms)
                # P300 xuất hiện trễ hay sớm phản ánh tốc độ xử lý thông tin
                peak_idx = np.argmax(signal)
                row[f'Comp{comp_idx}_Latency'] = times[peak_idx]
                
                # 4. Energy/Variance: Tổng năng lượng tín hiệu
                row[f'Comp{comp_idx}_Energy'] = np.sum(signal ** 2)
            
            final_data.append(row)
            
    # Lưu kết quả
    if final_data:
        df = pd.DataFrame(final_data)
        df.to_csv(OUTPUT_CSV, index=False)
        print(f"\n✅ ĐÃ TẠO TÀI NGUYÊN DỮ LIỆU TỐI ƯU: {OUTPUT_CSV}")
        print(f"   Kích thước: {df.shape}")
        print(f"   Số features: {df.shape[1]-3}")
    else:
        print("❌ Thất bại.")


In [4]:
generate_optimal_data()

💎 KHỞI TẠO QUY TRÌNH TỐI ƯU HÓA DỮ LIỆU (xDAWN)
⚡ Đang tối ưu hóa: sub-01...
NOTE: pick_types() is a legacy function. New code should use inst.pick(...).
Estimating covariance using EMPIRICAL
Done.
Estimating covariance using EMPIRICAL
Done.
Estimating covariance using EMPIRICAL
Done.
Estimating covariance using EMPIRICAL
Done.
Transforming to Xdawn space
Zeroing out 124 Xdawn components
Inverse transforming to sensor space
Transforming to Xdawn space
Zeroing out 124 Xdawn components
Inverse transforming to sensor space
Transforming to Xdawn space
Zeroing out 124 Xdawn components
Inverse transforming to sensor space
⚡ Đang tối ưu hóa: sub-02...
NOTE: pick_types() is a legacy function. New code should use inst.pick(...).
Estimating covariance using EMPIRICAL
Done.
Estimating covariance using EMPIRICAL
Done.
Estimating covariance using EMPIRICAL
Done.
Estimating covariance using EMPIRICAL
Done.
Transforming to Xdawn space
Zeroing out 124 Xdawn components
Inverse transforming to sensor sp

In [5]:
# OSL DYNAMICS HMM PIPELINE 
import os
import pickle
import numpy as np
import pandas as pd
import warnings
from mne.preprocessing import Xdawn
from osl_dynamics.models.hmm import Config, Model
from osl_dynamics.data import Data

# Tắt warning
warnings.filterwarnings("ignore")

# --- CẤU HÌNH ---
INPUT_EPOCHS = "/kaggle/input/data-processing/processed/processed_data.pkl"
OUTPUT_FILE = "/kaggle/working/osl_xdawn_tde_features.csv"

# Cấu hình TDE-HMM
N_STATES = 6           
N_EMBEDDINGS = 5       
PCA_COMPONENTS = 3     
SEQUENCE_LENGTH = 32   
BATCH_SIZE = 16        
N_EPOCHS_TRAIN = 20    

# Cửa sổ P300
WIN_TMIN = 0.25  
WIN_TMAX = 0.60  

def run_ultimate_pipeline():
    print(f"💎 KHỞI ĐỘNG OSL-DYNAMICS (FINAL STABLE)")
    print(f"   • Strategy: xDAWN Transform (127 -> 3) -> TDE (3 -> 15)")
    
    target_path = INPUT_EPOCHS
    if not os.path.exists(target_path):
        # Fallback path
        fb_path = "/kaggle/input/data-processing/processed_data.pkl"
        if os.path.exists(fb_path): target_path = fb_path
        else: print("Không tìm thấy file input."); return

    print(f"📂 Load dữ liệu từ: {target_path}")
    with open(target_path, 'rb') as f:
        all_subjects = pickle.load(f)

    clean_data_list = [] 
    meta_info = []       
    
    print(f"\n[PHẦN 1] TIỀN XỬ LÝ (xDAWN TRANSFORM)...")
    
    for subj_id, epochs in all_subjects.items():
        n_targets = np.sum(epochs.events[:, 2] == 6)
        if n_targets < 5: continue
            
        print(f"   ⚡ Xử lý {subj_id}...")
        
        # Chỉ lấy EEG & Resample 100Hz
        epochs.pick_types(eeg=True)
        if epochs.info['sfreq'] > 100:
            epochs.resample(100, verbose=False)
            
        # --- xDAWN SPATIAL FILTER ---
        xdawn = Xdawn(n_components=PCA_COMPONENTS, correct_overlap=False)
        try:
            # Fit trên toàn bộ dữ liệu để học spatial filters
            xdawn.fit(epochs)
            
            # Lấy epochs Target (6) và Standard (5) để trích xuất
            epochs_subset = epochs['Stimulus/S  5', 'Stimulus/S  6']
            
            X_trans = xdawn.transform(epochs_subset)
            X = X_trans[:, :PCA_COMPONENTS, :]
            
            events = epochs_subset.events[:, 2]
            times = epochs_subset.times
            
        except Exception as e:
            print(f"      ⚠️ Lỗi xDAWN {subj_id}: {e}. Bỏ qua.")
            continue
            
        print(f"      ✓ Output Shape: {X.shape}")

        # Index P300
        t_idx_start = np.abs(times - WIN_TMIN).argmin()
        t_idx_end = np.abs(times - WIN_TMAX).argmin()
        
        for i in range(len(X)):
            label = 1 if events[i] == 6 else 0
            
            # Transpose: (3, n_times) -> (n_times, 3)
            epoch_ts = X[i].T.astype(np.float32)
            
            clean_data_list.append(epoch_ts)
            
            meta_info.append({
                'subject_id': subj_id,
                'epoch_idx': i,
                'label': label,
                't_start': t_idx_start,
                't_end': t_idx_end
            })

    # --- TDE & TRAINING ---
    print(f"\n[PHẦN 2] OSL-DYNAMICS PIPELINE...")
    
    training_data = Data(clean_data_list)
    
    # TDE Embedding: 3 kênh -> 15 kênh
    print(f"   ⏳ Applying TDE (lags={N_EMBEDDINGS})...")
    training_data.tde(n_embeddings=N_EMBEDDINGS)
    training_data.standardize()
    
    print(f"Input channels cho HMM: {training_data.n_channels}")

    # Config Model 
    config = Config(
        n_states=N_STATES,
        n_channels=training_data.n_channels,
        sequence_length=SEQUENCE_LENGTH,
        learn_means=True,
        learn_covariances=True,
        batch_size=BATCH_SIZE,
        learning_rate=0.01,
        n_epochs=N_EPOCHS_TRAIN
    )
    
    print(f"Training HMM...")
    model = Model(config)
    model.fit(training_data)
    
    # --- INFERENCE ---
    print(f"\n[PHẦN 3] EXTRACTING FEATURES...")
    
    alphas = model.get_alpha(training_data)
    
    final_features = []
    t_offset = N_EMBEDDINGS // 2
    
    for i, alpha in enumerate(alphas):
        meta = meta_info[i]
        
        t_start_adj = max(0, meta['t_start'] - t_offset)
        t_end_adj = min(alpha.shape[0], meta['t_end'] - t_offset)
        
        if t_start_adj >= t_end_adj: continue
            
        alpha_window = alpha[t_start_adj:t_end_adj, :]
        
        if alpha_window.size == 0:
            fo = np.zeros(N_STATES)
        else:
            fo = np.mean(alpha_window, axis=0)
        
        row = {
            'subject_id': meta['subject_id'],
            'epoch_idx': meta['epoch_idx'],
            'label': meta['label']
        }
        for s in range(N_STATES):
            row[f'State_{s}_FO'] = fo[s]
            
        final_features.append(row)
            
    if final_features:
        df = pd.DataFrame(final_features)
        
        if df.isnull().values.any():
            df = df.fillna(0)
            
        df.to_csv(OUTPUT_FILE, index=False)
        print(f"\n✅ SUCCESS! File lưu tại: {OUTPUT_FILE}")
        print(f"   Kích thước: {df.shape}")
        
        print("\n📊 Average State Occupancy (Target vs Standard):")
        print(df.groupby('label')[[f'State_{s}_FO' for s in range(N_STATES)]].mean())
    else:
        print("❌ Failed.")



2025-11-27 07:30:59.010567: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1764228659.208423      21 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1764228659.258817      21 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

In [6]:
run_ultimate_pipeline()

💎 KHỞI ĐỘNG OSL-DYNAMICS (FINAL STABLE)
   • Strategy: xDAWN Transform (127 -> 3) -> TDE (3 -> 15)
📂 Load dữ liệu từ: /kaggle/input/data-processing/processed/processed_data.pkl

[PHẦN 1] TIỀN XỬ LÝ (xDAWN TRANSFORM)...
   ⚡ Xử lý sub-01...
NOTE: pick_types() is a legacy function. New code should use inst.pick(...).
Estimating covariance using EMPIRICAL
Done.
Estimating covariance using EMPIRICAL
Done.
Estimating covariance using EMPIRICAL
Done.
Estimating covariance using EMPIRICAL
Done.
      ✓ Output Shape: (72, 3, 100)
   ⚡ Xử lý sub-02...
NOTE: pick_types() is a legacy function. New code should use inst.pick(...).
Estimating covariance using EMPIRICAL
Done.
Estimating covariance using EMPIRICAL
Done.
Estimating covariance using EMPIRICAL
Done.
Estimating covariance using EMPIRICAL
Done.
      ✓ Output Shape: (43, 3, 100)
   ⚡ Xử lý sub-03...
NOTE: pick_types() is a legacy function. New code should use inst.pick(...).
Estimating covariance using EMPIRICAL
Done.
Estimating covariance

Loading files:   0%|          | 0/847 [00:00<?, ?it/s]

   ⏳ Applying TDE (lags=5)...


TDE:   0%|          | 0/847 [00:00<?, ?it/s]

Standardize:   0%|          | 0/847 [00:00<?, ?it/s]

...

 25/159 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - ll_loss: 20.1158 - loss: 20.1158

I0000 00:00:1764228705.487704      73 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


159/159 ━━━━━━━━━━━━━━━━━━━━ 15s 19ms/step - ll_loss: 17.9964 - loss: 17.9964 - learning_rate: 0.0100 - rho: 0.2853
Epoch 2/20
159/159 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - ll_loss: 16.0684 - loss: 16.0644 - learning_rate: 0.0090 - rho: 0.1866
Epoch 3/20
159/159 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - ll_loss: 15.9274 - loss: 15.9268 - learning_rate: 0.0082 - rho: 0.1436
Epoch 4/20
159/159 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - ll_loss: 15.8747 - loss: 15.8716 - learning_rate: 0.0074 - rho: 0.1187
Epoch 5/20
159/159 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - ll_loss: 15.9084 - loss: 15.9057 - learning_rate: 0.0067 - rho: 0.1022
Epoch 6/20
159/159 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - ll_loss: 15.9543 - loss: 15.9540 - learning_rate: 0.0061 - rho: 0.0904
Epoch 7/20
159/159 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - ll_loss: 15.9494 - loss: 15.9461 - learning_rate: 0.0055 - rho: 0.0814
Epoch 8/20
159/159 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - ll_loss: 15.9592 - loss: 15.9586 - learning_rate: 0.0050 - rho: 0.0743
Epoch 9/2

Getting alpha:   0%|          | 0/847 [00:00<?, ?it/s]


✅ SUCCESS! File lưu tại: /kaggle/working/osl_xdawn_tde_features.csv
   Kích thước: (847, 9)

📊 Average State Occupancy (Target vs Standard):
       State_0_FO  State_1_FO  State_2_FO  State_3_FO  State_4_FO  State_5_FO
label                                                                        
0        0.190066    0.089124    0.184544    0.190467    0.062967    0.282832
1        0.201500    0.177151    0.305822    0.099802    0.070449    0.145276


In [7]:
# extract feature using raw filtered
import os
import pickle
import numpy as np
import pandas as pd
import mne
from mne.preprocessing import Xdawn
import warnings

# OSL Library
try:
    from osl_dynamics.models.hmm import Config, Model
    from osl_dynamics.data import Data
except ImportError:
    raise ImportError("❌ Cần cài đặt thư viện 'osl-dynamics'")

warnings.filterwarnings("ignore")
DEFAULT_EPOCHS_PATH = "/kaggle/input/data-processing/processed/processed_data.pkl"

RAW_FILES = {
    "sub-01": "/kaggle/input/data-processing/processed/raw_filtered/sub-01_raw_filtered.fif",
    "sub-02": "/kaggle/input/data-processing/processed/raw_filtered/sub-02_raw_filtered.fif",
    "sub-03": "/kaggle/input/data-processing/processed/raw_filtered/sub-03_raw_filtered.fif",
    "sub-04": "/kaggle/input/data-processing/processed/raw_filtered/sub-04_raw_filtered.fif",
    "sub-05": "/kaggle/input/data-processing/processed/raw_filtered/sub-05_raw_filtered.fif"
}

OUTPUT_FILE = "./osl_raw_continuous_features.csv"

# Cấu hình
N_STATES = 6
N_EMBEDDINGS = 5       
PCA_COMPONENTS = 3     
SEQUENCE_LENGTH = 100  
BATCH_SIZE = 32
N_EPOCHS_TRAIN = 30    

WIN_TMIN = 0.25
WIN_TMAX = 0.60

def sanitize_data(data):
    if np.isnan(data).any(): data = np.nan_to_num(data)
    std = np.std(data, axis=0)
    mean = np.mean(data, axis=0)
    return np.clip(data, mean - 20*std, mean + 20*std)

def run_final_pipeline():
    print(f"🚀 BẮT ĐẦU: FINAL PIPELINE (WITH EPOCH_ID)")
    
    target_path = DEFAULT_EPOCHS_PATH
    if not os.path.exists(target_path):
        alt = "/kaggle/input/data-processing/processed_data.pkl"
        if os.path.exists(alt): target_path = alt
        else: print("❌ Thiếu file Epochs."); return

    print(f"📂 Epochs: {target_path}")
    with open(target_path, 'rb') as f:
        ref_subjects = pickle.load(f)

    continuous_data_list = []
    subject_meta = []

    print(f"\n[BƯỚC 1] CHIẾU VÀ GIẢM CHIỀU DỮ LIỆU...")

    for subj_id, raw_path in RAW_FILES.items():
        if subj_id not in ref_subjects: continue
        epochs_ref = ref_subjects[subj_id]
        
        if np.sum(epochs_ref.events[:, 2] == 6) < 5: continue
        if not os.path.exists(raw_path): continue
            
        print(f"   ⚡ {subj_id}...", end=" ")
        
        try:
            raw = mne.io.read_raw_fif(raw_path, preload=True, verbose='error')
        except: continue

        raw.pick_types(eeg=True)
        if raw.info['sfreq'] > 100: raw.resample(100, verbose=False)
        if epochs_ref.info['sfreq'] > 100: epochs_ref = epochs_ref.copy().resample(100, verbose=False)

        # --- MANUAL PROJECTION ---
        try:
            xdawn = Xdawn(n_components=PCA_COMPONENTS, correct_overlap=False)
            xdawn.fit(epochs_ref)
            
            all_filters = xdawn.filters_['Stimulus/S  6']
            best_filters = all_filters[:PCA_COMPONENTS, :] 
            
            raw_data = raw.get_data()
            projected_data = np.dot(best_filters, raw_data)
            
            print(f"Done. Shape: {projected_data.shape}")
            
        except Exception as e:
            print(f"❌ Lỗi: {e}"); continue

        data_final = projected_data.T.astype(np.float32)
        data_final = sanitize_data(data_final)
        
        continuous_data_list.append(data_final)
        
        subject_meta.append({
            'sid': subj_id,
            'epoch_events': epochs_ref.events, 
            'sfreq': raw.info['sfreq']
        })

    # --- OSL PIPELINE ---
    print(f"\n[BƯỚC 2] HUẤN LUYỆN HMM...")
    
    if not continuous_data_list: print("❌ Không có data."); return

    training_data = Data(continuous_data_list)
    training_data.tde(n_embeddings=N_EMBEDDINGS)
    training_data.standardize()
    
    n_chals = training_data.n_channels
    print(f"   📊 Input Channels: {n_chals}")
    
    if n_chals > 20: return

    config = Config(
        n_states=N_STATES,
        n_channels=n_chals,
        sequence_length=SEQUENCE_LENGTH,
        learn_means=True,
        learn_covariances=True,
        batch_size=BATCH_SIZE,
        learning_rate=0.01,
        n_epochs=N_EPOCHS_TRAIN
    )
    
    model = Model(config)
    model.fit(training_data)
    
    # --- INFERENCE ---
    print(f"\n[BƯỚC 3] TRÍCH XUẤT (CÓ EPOCH_ID)...")
    
    alphas = model.get_alpha(training_data)
    final_features = []
    t_offset = N_EMBEDDINGS // 2 
    
    for idx, alpha_cont in enumerate(alphas):
        meta = subject_meta[idx]
        events = meta['epoch_events']
        sfreq = meta['sfreq']
        
        win_start = int(WIN_TMIN * sfreq)
        win_end = int(WIN_TMAX * sfreq)
        
        for i in range(len(events)):
            evt_time = events[i, 0]
            code = events[i, 2]
            
            if code == 6: label = 1
            elif code == 5: label = 0
            else: continue
            
            start = evt_time + win_start - t_offset
            end = evt_time + win_end - t_offset
            
            if start < 0 or end > alpha_cont.shape[0]: continue
            
            alpha_win = alpha_cont[start:end, :]
            if alpha_win.size == 0: continue
            
            fo = np.mean(alpha_win, axis=0)
            
            # --- ĐÃ THÊM EPOCH_IDX TẠI ĐÂY ---
            row = {
                'subject_id': meta['sid'],
                'epoch_idx': i,  # <-- Cột quan trọng để định danh epoch
                'label': label
            }
            for s in range(N_STATES):
                row[f'State_{s}_FO'] = fo[s]
            final_features.append(row)

    if final_features:
        df = pd.DataFrame(final_features).fillna(0)
        df.to_csv(OUTPUT_FILE, index=False)
        print(f"\n✅ HOÀN TẤT! File: {OUTPUT_FILE}")
        
        print("\n🔍 Kiểm tra nhanh (Mean per Class):")
        print(df.groupby('label').mean(numeric_only=True))
    else:
        print("❌ Thất bại.")

In [8]:
run_final_pipeline()

🚀 BẮT ĐẦU: FINAL PIPELINE (WITH EPOCH_ID)
📂 Epochs: /kaggle/input/data-processing/processed/processed_data.pkl

[BƯỚC 1] CHIẾU VÀ GIẢM CHIỀU DỮ LIỆU...
   ⚡ sub-01... NOTE: pick_types() is a legacy function. New code should use inst.pick(...).
Estimating covariance using EMPIRICAL
Done.
Estimating covariance using EMPIRICAL
Done.
Estimating covariance using EMPIRICAL
Done.
Estimating covariance using EMPIRICAL
Done.
Done. Shape: (3, 137132)
   ⚡ sub-02... NOTE: pick_types() is a legacy function. New code should use inst.pick(...).
Estimating covariance using EMPIRICAL
Done.
Estimating covariance using EMPIRICAL
Done.
Estimating covariance using EMPIRICAL
Done.
Estimating covariance using EMPIRICAL
Done.
Done. Shape: (3, 134582)
   ⚡ sub-03... NOTE: pick_types() is a legacy function. New code should use inst.pick(...).
Estimating covariance using EMPIRICAL
Done.
Estimating covariance using EMPIRICAL
Done.
Estimating covariance using EMPIRICAL
Done.
Estimating covariance using EMPIRICAL


Loading files:   0%|          | 0/4 [00:00<?, ?it/s]

TDE:   0%|          | 0/4 [00:00<?, ?it/s]

Standardize:   0%|          | 0/4 [00:00<?, ?it/s]

   📊 Input Channels: 15
Epoch 1/30
168/168 ━━━━━━━━━━━━━━━━━━━━ 12s 24ms/step - ll_loss: 15.2384 - loss: 15.2398 - learning_rate: 0.0100 - rho: 0.2853
Epoch 2/30
168/168 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - ll_loss: 11.0304 - loss: 11.0307 - learning_rate: 0.0090 - rho: 0.2094
Epoch 3/30
168/168 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - ll_loss: 10.5184 - loss: 10.5193 - learning_rate: 0.0082 - rho: 0.1691
Epoch 4/30
168/168 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - ll_loss: 10.3563 - loss: 10.3565 - learning_rate: 0.0074 - rho: 0.1436
Epoch 5/30
168/168 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - ll_loss: 10.5235 - loss: 10.5205 - learning_rate: 0.0067 - rho: 0.1258
Epoch 6/30
168/168 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - ll_loss: 10.4160 - loss: 10.4149 - learning_rate: 0.0061 - rho: 0.1125
Epoch 7/30
168/168 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - ll_loss: 10.2300 - loss: 10.2295 - learning_rate: 0.0055 - rho: 0.1022
Epoch 8/30
168/168 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - ll_loss: 10.1283 - loss: 10.1283 - learning_r

Getting alpha:   0%|          | 0/4 [00:00<?, ?it/s]


✅ HOÀN TẤT! File: ./osl_raw_continuous_features.csv

🔍 Kiểm tra nhanh (Mean per Class):
       epoch_idx  State_0_FO    State_1_FO  State_2_FO  State_3_FO  \
label                                                                
0      32.000000    0.350837  1.286077e-02    0.166641    0.052864   
1      28.190476    0.418970  3.578025e-09    0.207425    0.040566   

       State_4_FO  State_5_FO  
label                          
0        0.090785    0.326012  
1        0.088258    0.244781  
